In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle

customers = pd.read_csv("../data/customers.csv")
articles = pd.read_csv("../data/articlesb.csv")
transactions = pd.read_csv("../data/transactions_train.csv")
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])


In [2]:
sample_transactions = transactions.sample(n=100000, random_state=42)

positives = sample_transactions[["customer_id", "article_id", "price"]].copy()
positives["label"] = 1

positives.head()


,customer_id,article_id,price,label
16558544,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,0.022017,1
6583409,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,0.028797,1
13976622,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,0.050831,1
10327778,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,0.027102,1
15285658,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,0.050831,1


In [3]:
all_articles = articles["article_id"].values
negative_rows = []

for _, row in positives.sample(n=50000, random_state=42).iterrows():
    negative_rows.append({
        "customer_id": row["customer_id"],
        "article_id": np.random.choice(all_articles),
        "price": row["price"],
        "label": 0
    })

negatives = pd.DataFrame(negative_rows)

training_data = pd.concat([positives, negatives], ignore_index=True)

training_data["label"].value_counts(normalize=True)


label
1    0.666667
0    0.333333
Name: proportion, dtype: float64

In [4]:
user_features = transactions.groupby("customer_id").agg(
    user_total_purchases=("article_id", "count"),
    user_avg_price=("price", "mean"),
    user_unique_articles=("article_id", "nunique")
).reset_index()

article_features = transactions.groupby("article_id").agg(
    article_total_purchases=("customer_id", "count"),
    article_unique_buyers=("customer_id", "nunique"),
    article_avg_price=("price", "mean")
).reset_index()

df = training_data.merge(user_features, on="customer_id", how="left")
df = df.merge(article_features, on="article_id", how="left")
df = df.merge(
    articles[["article_id", "product_type_name", "department_name", "colour_group_name"]],
    on="article_id",
    how="left"
)

df = df.fillna(0)

df.head()

,customer_id,article_id,price,label,user_total_purchases,user_avg_price,user_unique_articles,article_total_purchases,article_unique_buyers,article_avg_price,product_type_name,department_name,colour_group_name
0,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,0.022017,1,14,0.022804,13,119.0,105.0,0.019302,Leggings/Tights,Young Girl Jersey Fancy,Black
1,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,0.028797,1,154,0.027369,134,101.0,85.0,0.041454,Pyjama set,Nightwear,Dark Blue
2,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,0.050831,1,101,0.027324,81,561.0,459.0,0.047309,Swimsuit,Swimwear,Dark Orange
3,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,0.027102,1,11,0.020647,11,852.0,788.0,0.027068,Blouse,Blouse,Dark Green
4,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,0.050831,1,119,0.043282,90,422.0,338.0,0.045926,Trousers,Trouser,White


In [5]:
categorical_cols = ["product_type_name", "department_name", "colour_group_name"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    

In [6]:
feature_cols = [
    "price",
    "user_total_purchases",
    "user_avg_price",
    "user_unique_articles",
    "article_total_purchases",
    "article_unique_buyers",
    "article_avg_price",
    "product_type_name",
    "department_name",
    "colour_group_name"
]

X = df[feature_cols]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Train score:", model.score(X_train, y_train))
print("Test score:", model.score(X_test, y_test))

Train score: 0.871875
Test score: 0.8625666666666667


In [7]:
pickle.dump(model, open("../data/ranking_model.pkl", "wb"))
pickle.dump(feature_cols, open("../data/feature_cols.pkl", "wb"))


In [8]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

importance

,feature,importance
4,article_total_purchases,0.420893
5,article_unique_buyers,0.370960
6,article_avg_price,0.080841
0,price,0.061312
8,department_name,0.026348
2,user_avg_price,0.011793
9,colour_group_name,0.009603
7,product_type_name,0.006861
1,user_total_purchases,0.005738
3,user_unique_articles,0.005651
